Build the RISCV target

In [1]:
import tvm
import tvm.relax as relax
from tvm import meta_schedule as ms
from tvm.script import relax as R
from tvm.script import tir as T

target = tvm.target.Target(
    "llvm -mtriple=riscv64-unknown-linux-gnu "
    "-mattr=+64bit,+m,+a,+f,+d,+c,+v "
    "-mcpu=generic-rv64 "
    "-mabi=lp64d "
)

Register intrinsics

In [2]:
from tvm.tir.tensor_intrin.riscv_cpu import (
    register_riscv_intrinsics,
    get_max_elems,
)
from tvm.runtime import DataType
from tvm.target.codegen import llvm_get_vector_width

BATCH, CHANNELS, H, W = 14, 23, 67, 99
SHAPE = (BATCH, CHANNELS, H, W)
DTYPE = "float32"

In [10]:
@tvm.script.ir_module
class AddModule:
    @R.function
    def main(x: R.Tensor(SHAPE, DTYPE), y: R.Tensor(SHAPE, DTYPE)):
        with R.dataflow():
            gv = relax.op.add(x, y)
            R.output(gv)
        return gv

In [4]:
with target:
    inventory = register_riscv_intrinsics(target)

vlen      = llvm_get_vector_width(target)
dt        = DataType(DTYPE)
max_elems = get_max_elems(vlen, lmul=8, sew=dt.bits)

n_elems   = min(SHAPE[0], max_elems)

intrin_name = "rvv_add_32u8_4x32i8_4i32"
intrin = tvm.tir.TensorIntrin.get(intrin_name)

In [11]:
#mod = tvm.IRModule({"main": intrin.impl})
mod = AddModule

mod.show()

In [12]:
ex = tvm.compile(mod, target=target)

In [13]:
import os
OUTPUT_DIR = "output/intrinsics"

ex = tvm.compile(mod, target=target)

so_path = os.path.join(OUTPUT_DIR, f"intrinsic_add.so")
asm_path = os.path.join(OUTPUT_DIR, f"intrinsic_add.asm")

ex.export_library(
    so_path,
    cc="riscv64-linux-gnu-gcc"
)
print(f"    [INFO] Shared lib -> {so_path}")

import subprocess
result = subprocess.run(
    ["llvm-objdump-18", "-d", "--mattr=+v", so_path],
    capture_output=True, text=True, check=True
)
with open(asm_path, "w") as f:
    f.write(result.stdout)
print(f"    [INFO] Disassembly -> {asm_path}")

    [INFO] Shared lib -> output/intrinsics/intrinsic_add.so
    [INFO] Disassembly -> output/intrinsics/intrinsic_add.asm
